# Simultaneous Fitting: several datasets, one answer

Contrast variation, a temperature series, one sample measured on three detector
distances — these all have the same shape. Most of the physics is common to
every curve, a few parameters are not, and fitting the curves one at a time
throws away the constraint that makes the common part identifiable.

`MultiFitter` fits them together: several named datasets, each with its own
model, Q range, resolution and parameters, minimised as **one problem** with
parameters shared or related between them.

This notebook works through:

1. [Two contrasts, fitted separately](#separately) — what one curve can say on its own
2. [The same two curves, fitted together](#together) — sharing what is physically shared
3. [Reading the result](#reading) — statuses, roots and uncertainties
4. [Plots and export](#plots)
5. [Constraints](#constraints) — constants, equality and arithmetic
6. [What a constraint refuses to do](#refuses) — and why that is the point
7. [Three instrument configurations](#configurations) — different Q ranges, no merging
8. [Dataset weights](#weights) — what they change, and what they do not
9. [Choosing what to share](#choosing)

Everything below uses simulated data with a known answer, so each fitted number
can be checked against the truth it came from.

In [ ]:
import numpy as np

import sans_fitter
from sans_fitter import MultiFitter, SANSFitter, examples

# The fitter narrates what it does; the point of most cells below is the
# summary they print themselves, so keep the running commentary quiet.
sans_fitter.set_verbosity('warning')

TRUE_RADIUS = 48.0
TRUE_PD = 0.12
TRUE_SCALE = 0.02
TRUE_SLD = 4.0
SOLVENTS = {'h2o': -0.56, 'd2o': 6.34}
BACKGROUNDS = {'h2o': 0.012, 'd2o': 0.035}

One particle, two solvents. The size, the size distribution and the
concentration are properties of the **sample**, so they are identical in both
measurements. The solvent SLD is the variable being changed, and the incoherent
background follows the solvent, so those differ.

In [ ]:
datasets = {
    name: examples.simulate(
        'sphere',
        radius=TRUE_RADIUS,
        radius_pd=TRUE_PD,
        scale=TRUE_SCALE,
        background=BACKGROUNDS[name],
        sld=TRUE_SLD,
        sld_solvent=SOLVENTS[name],
        qmin=0.006,
        qmax=0.4,
        npoints=70,
        noise=0.03,
        seed=seed,
    )
    for seed, name in enumerate(SOLVENTS, start=101)
}

for name, data in datasets.items():
    print(f'{name}: {len(data.x)} points, '
          f'Q {data.x.min():.4f}-{data.x.max():.3f}, '
          f'solvent SLD {SOLVENTS[name]}')

<a id="separately"></a>
## 1. Two contrasts, fitted separately

First, the baseline: what each curve says on its own.

In [ ]:
def fit_one(name):
    fitter = SANSFitter()
    fitter.set_data(datasets[name])
    fitter.set_model('sphere')
    fitter.set_param('radius', value=35, min=10, max=120, vary=True)
    fitter.set_param('scale', value=0.01, min=0.001, max=0.2, vary=True)
    fitter.set_param('background', value=0.02, min=0.0, max=0.2, vary=True)
    fitter.set_param('sld', value=TRUE_SLD, vary=False)
    fitter.set_param('sld_solvent', value=SOLVENTS[name], vary=False)
    fitter.enable_polydispersity(True)
    fitter.set_pd_param('radius', pd_width=0.05, vary=True)
    return fitter.fit(engine='bumps', method='lm')


separate = {name: fit_one(name) for name in datasets}

print(f'{"dataset":<8} {"radius":>14} {"radius_pd":>14} {"scale":>16}')
print('-' * 56)
for name, result in separate.items():
    p = result['parameters']
    print(f'{name:<8} {p["radius"]["formatted"]:>14} '
          f'{p["radius_pd"]["formatted"]:>14} {p["scale"]["formatted"]:>16}')
print(f'\ntruth:   radius {TRUE_RADIUS}, radius_pd {TRUE_PD}, scale {TRUE_SCALE}')

Two radii, two size distributions, two concentrations — for one sample. Nothing
stops them from disagreeing, because nothing told the fit they were the same
particle.

<a id="together"></a>
## 2. The same two curves, fitted together

`add()` registers a named dataset and returns a handle for configuring it.
`fit['h2o']` reaches the same handle later.

Two things to note in the setup below:

- **The starting values are identical in both datasets.** `share()` is
  symmetric, so the members have to agree before it will make them one quantity.
  Which dataset's starting radius the fit uses is a scientific choice, not a
  consequence of the order two files were loaded in — so a disagreement is an
  error rather than a silent pick. (`share(..., source='h2o')` decides it
  explicitly when they genuinely differ.)
- **The solvent SLDs are constrained, not fitted.** They were measured.

In [ ]:
fit = MultiFitter()

for name in datasets:
    fit.add(name, datasets[name], model='sphere')
    entry = fit[name]
    entry.set_param('radius', value=35, min=10, max=120, vary=True)
    entry.set_param('scale', value=0.01, min=0.001, max=0.2, vary=True)
    entry.set_param('background', value=0.02, min=0.0, max=0.2, vary=True)
    entry.set_param('sld', value=TRUE_SLD, vary=False)
    entry.enable_polydispersity(True)
    entry.set_pd_param('radius', pd_width=0.05, pd_type='gaussian', vary=True)
    fit.constrain(f'{name}.sld_solvent', SOLVENTS[name])

# The sample is the same in both solvents. The background is not.
fit.share('radius', 'radius_pd', 'scale')

`describe()` is the thing to read before fitting. The number that matters is the
free-parameter count: sharing is supposed to reduce it, and that is the easiest
thing to get wrong.

In [ ]:
fit.describe()

Eight configured parameters across two datasets have become **five** free ones:
one radius, one width, one scale, and the two backgrounds that really are
separate. The two solvent SLDs are constants and cost nothing.

`plot_model()` shows the starting point with every relationship already applied
— it previews the constrained model, not whatever the individual parameter
tables happen to say.

In [ ]:
fit.plot_model(show=False)

In [ ]:
result = fit.fit(method='lm')

In [ ]:
print(fit.get_fit_report())

<a id="reading"></a>
## 3. Reading the result

Against the truth the data was generated from:

In [ ]:
checks = [
    ('radius', result.parameters['d2o.radius'], TRUE_RADIUS),
    ('radius_pd', result.parameters['d2o.radius_pd'], TRUE_PD),
    ('scale', result.parameters['d2o.scale'], TRUE_SCALE),
    ('h2o.background', result.parameters['h2o.background'], BACKGROUNDS['h2o']),
    ('d2o.background', result.parameters['d2o.background'], BACKGROUNDS['d2o']),
]

print(f'{"parameter":<16} {"fitted":>16} {"truth":>10}   deviation')
print('-' * 58)
for label, entry, truth in checks:
    sigma = abs(entry.value - truth) / entry.stderr
    print(f'{label:<16} {entry.formatted:>16} {truth:>10g}   {sigma:.1f} sigma')

Compare the shared quantities with the separate fits from section 1: the joint
error bars are smaller, because two curves constrain one radius.

In [ ]:
print(f'{"quantity":<12} {"h2o alone":>14} {"d2o alone":>14} {"joint":>16}')
print('-' * 60)
for key in ('radius', 'radius_pd', 'scale'):
    joint = result.parameters[f'd2o.{key}']
    print(f'{key:<12} {separate["h2o"]["parameters"][key]["formatted"]:>14} '
          f'{separate["d2o"]["parameters"][key]["formatted"]:>14} '
          f'{joint.formatted:>16}')

### Statuses, roots and members

Every parameter carries a **status** and a **root**. A shared radius is one
quantity with two names, so it must be counted — and reported — once:

| status | meaning | uncertainty |
|---|---|---|
| `free` | an independent coordinate the optimizer moved | from the joint covariance |
| `shared` | the same quantity as one or more others | the same as its root's |
| `derived` | computed from others by a constraint | propagated through the constraint |
| `fixed` | not fitted, or pinned by a constant | exactly zero |

`result.root_parameters()` gives one entry per distinct quantity;
`result.parameters` gives every name.

In [ ]:
print(f'{"parameter":<18} {"status":<8} {"root":<14} {"members"}')
print('-' * 74)
for entry in result.parameters.values():
    members = ', '.join(entry.members) if len(entry.members) > 1 else ''
    print(f'{entry.qualified:<18} {entry.status:<8} {entry.root:<14} {members}')

In [ ]:
# A shared member is not an extra coordinate: the two names are one number.
h2o_radius = result.parameters['h2o.radius']
d2o_radius = result.parameters['d2o.radius']
print('same value: ', h2o_radius.value == d2o_radius.value)
print('same stderr:', h2o_radius.stderr == d2o_radius.stderr)
print('free parameters:', result.n_free, '->', result.cov_labels)

### Per-dataset diagnostics

Each dataset reports its own point count and χ², but **not** a reduced χ² — the
degrees of freedom belong to the joint fit and cannot be divided between
datasets. The per-point figure that *can* be compared is χ²/N, labelled as a
mean squared normalized residual.

In [ ]:
print(f'{"dataset":<8} {"points":>7} {"chi2":>9} {"chi2/N":>8} {"weight":>7}  resolution')
print('-' * 72)
for entry in result.datasets.values():
    print(f'{entry.name:<8} {entry.n_points:>7} {entry.chisq:>9.2f} '
          f'{entry.mean_squared_residual:>8.3f} {entry.weight:>7g}  {entry.resolution}')

print(f'\njoint: {result.n_points} points, {result.n_free} free, '
      f'dof {result.dof}, chi2/dof {result.reduced_chisq:.3f}')

<a id="plots"></a>
## 4. Plots and export

One panel per dataset, each with its own axes and its own residuals underneath.
Stacking rather than overlaying is the default because datasets in a joint fit
routinely differ by orders of magnitude in intensity — which is often *why* they
are being fitted together.

In [ ]:
fit.plot_results(show=False)

In [ ]:
import tempfile
import os

output = os.path.join(tempfile.mkdtemp(), 'contrast_results')
fit.save_results(output)
print('\n'.join(sorted(os.listdir(output))))

The exported contributions reconstruct the reported totals exactly — summing the
squared residuals across the curve files gives back `chisq`:

In [ ]:
def read_curve(path):
    lines = [line for line in open(path, encoding='utf-8').read().splitlines()
             if line and not line.startswith('#')]
    columns = lines[0].split(',')
    values = np.array([[float(c) for c in line.split(',')] for line in lines[1:]])
    return dict(zip(columns, values.T))


total = sum(
    float(np.sum(read_curve(os.path.join(output, f'{name}_curve.csv'))['Residual'] ** 2))
    for name in datasets
)
print(f'sum of squared exported residuals: {total:.6f}')
print(f'reported chisq:                    {result.chisq:.6f}')

<a id="constraints"></a>
## 5. Constraints: constants, equality and arithmetic

Three relationships are available, and they mean different things.

**`share()`** — symmetric. These are one quantity; the members must agree, and
their bounds intersect so a limit set on any member still applies.

**`link_params(a, to=b)`** — directed. `a` follows `b`, adopting `b`'s value,
vary flag and bounds. Use it when the parameters have different names or live in
different models.

**`constrain()`** — a constant, a bare reference (equality), or arithmetic.

Here is a concentration series where the dilution ratio is known from the sample
preparation, so the diluted scale is not an independent parameter:

In [ ]:
series = MultiFitter()
truth = {'stock': 0.024, 'diluted': 0.012}

for seed, (name, scale) in enumerate(truth.items(), start=301):
    data = examples.simulate(
        'sphere', radius=TRUE_RADIUS, scale=scale, background=0.01,
        sld=TRUE_SLD, sld_solvent=6.34,
        qmin=0.006, qmax=0.35, npoints=60, noise=0.03, seed=seed,
    )
    series.add(name, data, model='sphere')
    entry = series[name]
    entry.set_param('radius', value=40, min=10, max=120, vary=True)
    entry.set_param('background', value=0.01, min=0.0, max=0.1, vary=True)
    entry.set_param('sld', value=TRUE_SLD, vary=False)
    series.constrain(f'{name}.sld_solvent', 6.34)

series.share('radius')
series['stock'].set_param('scale', value=0.02, min=0.001, max=0.1, vary=True)
series['diluted'].set_param('scale', min=0.0005, max=0.05)
series.constrain('diluted.scale', '0.5 * stock.scale')

series_result = series.fit(method='lm')

stock = series_result.parameters['stock.scale']
diluted = series_result.parameters['diluted.scale']
print(f'stock.scale    {stock.formatted:>16}   truth {truth["stock"]}   ({stock.status})')
print(f'diluted.scale  {diluted.formatted:>16}   truth {truth["diluted"]}   ({diluted.status})')
print(f'\nratio held exactly: {diluted.value / stock.value}')
print(f'derived error / root error: {diluted.stderr / stock.stderr}')
print(f'uncertainty source: {diluted.uncertainty_source}')

The derived error is the root's error times the coefficient — propagated through
the constraint, not estimated independently. For an expression reading more than
one parameter the propagation keeps the **cross-covariance**: `a - b` cannot use
independent-error quadrature unless the covariance really is zero.

The grammar is deliberately small — numbers, qualified references, `+ - * /` and
integer powers — and expressions are **interpreted, never executed**. The text is
parsed and walked against an allowlist, so a constraint is data rather than code.

In [ ]:
for bad in ["__import__('os')", 'abs(stock.scale)', 'scale', 'stock.scale ** 9']:
    try:
        series.constrain('diluted.background', bad)
    except ValueError as error:
        print(f'{bad!r}\n  -> {type(error).__name__}: {str(error)[:100]}...\n')

<a id="refuses"></a>
## 6. What a constraint refuses to do, and why

Binding an expression replaces the target's parameter object inside bumps, which
**discards the target's own limits**. Rather than let a limit quietly stop
applying, `MultiFitter` proves it before the fit: a constraint is accepted only
if it cannot leave the target's bounds over the ranges its inputs are allowed to
explore.

Where the relationship is linear in one parameter that proof is *constructive* —
the input's range is narrowed so the target is guaranteed:

In [ ]:
demo = MultiFitter()
for seed, name in enumerate(('a', 'b'), start=401):
    demo.add(name, examples.simulate('sphere', radius=45, noise=0.02,
                                     seed=seed, npoints=30), model='sphere')
    demo[name].set_param('scale', value=0.01, min=0.001, max=0.1, vary=True)

demo['b'].set_param('scale', min=0.0, max=0.03)   # the target's own limit
demo.constrain('b.scale', '2 * a.scale')

graph = demo._compiled()
root = next(c for c in graph.classes if c.label == 'a.scale')
print(f'a.scale configured range: 0.001 to 0.1')
print(f'a.scale fitted range:     {graph.ranges[root.index]}')
print('\nb.scale is bounded above by 0.03, so a.scale cannot exceed 0.015.')

Where the arithmetic **cannot** certify the bound — a divisor whose range
straddles zero, a nonlinear expression whose interval estimate is too loose —
the constraint is refused, and the message names what to tighten. This is
conservative: some feasible nonlinear relationships need tighter bounds than they
strictly must. It is preferred to a bound that silently stops being enforced.

In [ ]:
try:
    demo.constrain('b.background', '1 / a.background')
except ValueError as error:
    print(error)

The same protection applies to the analysis as a whole. A change that would
leave it inconsistent is rejected **and rolled back**, so the fitter is never in
a state that cannot be fitted:

In [ ]:
# 'h2o' takes part in several relationships, and the dab model has none of the
# parameters they name. Changing the model would orphan them, so it is refused.
print('before:', fit['h2o'].model_name,
      '| constraints:', [c['target'] for c in fit.get_constraints()])
try:
    fit['h2o'].set_model('dab')
except ValueError as error:
    print('refused:', error)
print('after:', fit['h2o'].model_name,
      '| constraints:', [c['target'] for c in fit.get_constraints()])

<a id="configurations"></a>
## 7. Three instrument configurations

Nothing here requires a common Q grid. Each dataset is evaluated on its own
points with its own resolution kernel; only the residual vectors are
concatenated. That is why fitting the original curves does **not** need data
merging or rebinning.

In [ ]:
settings = {
    'low_q': dict(qmin=0.003, qmax=0.03, dq=0.14, npoints=30),
    'mid_q': dict(qmin=0.02, qmax=0.15, dq=0.10, npoints=40),
    'high_q': dict(qmin=0.1, qmax=0.5, dq=0.06, npoints=35),
}

configs = MultiFitter()
for seed, (name, setting) in enumerate(settings.items(), start=201):
    data = examples.simulate(
        'sphere', radius=TRUE_RADIUS, radius_pd=TRUE_PD, scale=TRUE_SCALE,
        background=0.02, sld=TRUE_SLD, sld_solvent=6.34, noise=0.03, seed=seed,
        **setting,
    )
    configs.add(name, data, model='sphere')
    entry = configs[name]
    entry.set_param('radius', value=35, min=10, max=120, vary=True)
    entry.set_param('scale', value=0.02, min=0.001, max=0.2, vary=True)
    entry.set_param('background', value=0.02, min=0.0, max=0.2, vary=True)
    entry.set_param('sld', value=TRUE_SLD, vary=False)
    entry.enable_polydispersity(True)
    entry.set_pd_param('radius', pd_width=0.05, vary=True)
    entry.set_resolution('data')          # each carries its own simulated dQ
    configs.constrain(f'{name}.sld_solvent', 6.34)

# One sample: everything about the sample is shared. Only the background is
# per-configuration, because it depends on the setup.
configs.share('radius', 'radius_pd', 'scale')
config_result = configs.fit(method='lm')

print(f'{"configuration":<14} {"points":>7} {"Q range":>20} {"chi2/N":>9}')
print('-' * 54)
for entry in config_result.datasets.values():
    span = f'{entry.q_range[0]:.4g} - {entry.q_range[1]:.4g}'
    print(f'{entry.name:<14} {entry.n_points:>7} {span:>20} '
          f'{entry.mean_squared_residual:>9.3f}')

radius = config_result.parameters['high_q.radius']
print(f'\njoint radius: {radius.formatted}  (truth {TRUE_RADIUS})')

In [ ]:
configs.plot_results(show=False)

<a id="weights"></a>
## 8. Dataset weights

By default every dataset counts according to its own uncertainties, and the
total is a χ². A dataset with more points or smaller error bars legitimately
carries more information, and nothing rebalances that automatically.

`set_dataset_weight()` states a **fitting priority**. It changes which
compromise the optimizer prefers; it does not change the error model, because
the supplied `dI` is still what the uncertainties mean. Two things follow, and
the report labels both.

In [ ]:
weighted = MultiFitter()
for name in datasets:
    weighted.add(name, datasets[name], model='sphere')
    entry = weighted[name]
    entry.set_param('radius', value=35, min=10, max=120, vary=True)
    entry.set_param('scale', value=0.01, min=0.001, max=0.2, vary=True)
    entry.set_param('background', value=0.02, min=0.0, max=0.2, vary=True)
    entry.set_param('sld', value=TRUE_SLD, vary=False)
    weighted.constrain(f'{name}.sld_solvent', SOLVENTS[name])
weighted.share('radius', 'scale')
weighted.set_dataset_weight('d2o', 0.25)

weighted_result = weighted.fit(method='lm')

print(f'chisq      (goodness of fit) : {weighted_result.chisq:.3f}')
print(f'objective  (what was minimised): {weighted_result.objective:.3f}')
print(f'covariance source: {weighted_result.cov_source}')

**First**, `chisq` and `objective` become different numbers. The raw χ² at an
artificially weighted optimum is not at its own minimum, so it is no longer a
goodness-of-fit test in the usual sense — and the objective is not a χ² at all.

**Second**, uncertainties switch to the known-error *sandwich* covariance
`H⁻¹ (JᵀW²J) H⁻¹`. The inverse curvature of the reweighted objective would be the
right answer only if `dI/√a` were the real error bars, which is not what the
weights mean here. A property that tells the two apart: multiplying **every**
weight by one constant changes nothing about the fit, so it must not change the
uncertainties either.

In [ ]:
def covariance_for(scale):
    trial = MultiFitter()
    for name in datasets:
        trial.add(name, datasets[name], model='sphere')
        entry = trial[name]
        entry.set_param('radius', value=35, min=10, max=120, vary=True)
        entry.set_param('scale', value=0.01, min=0.001, max=0.2, vary=True)
        entry.set_param('background', value=0.02, min=0.0, max=0.2, vary=True)
        entry.set_param('sld', value=TRUE_SLD, vary=False)
        trial.constrain(f'{name}.sld_solvent', SOLVENTS[name])
    trial.share('radius', 'scale')
    trial.set_dataset_weight('h2o', 1.0 * scale)
    trial.set_dataset_weight('d2o', 0.25 * scale)
    return trial.fit(method='lm')


base = covariance_for(1.0)
rescaled = covariance_for(9.0)
print('same labels: ', base.cov_labels == rescaled.cov_labels)
print('same covariance:', np.allclose(base.cov, rescaled.cov, rtol=1e-5))

Weights must be positive and finite. To leave a dataset out, remove it rather
than weighting it to zero.

### The independence assumption

The joint χ² assumes independent Gaussian point errors. Overlapping Q ranges
from separate measurements are fine. Duplicate observations — or datasets sharing
a measured subtraction background — are not independent, and counting one
measurement twice makes the uncertainties optimistic. `MultiFitter` warns when
two datasets hold identical data.

<a id="choosing"></a>
## 9. Choosing what to share

Sharing is a physical claim, not a convenience. Same-named parameters are not
automatically the same quantity.

| parameter | share when |
|---|---|
| geometry (`radius`, `length`, `thickness`) | the particles are the same — the usual reason for a joint fit |
| polydispersity width | the size distribution is the same; requires the same distribution type on every entry |
| `scale` | the concentration **and** the normalization are the same — contrast variation with matched samples, yes; a concentration series, no |
| `sld` (particle) | the material is the same and the contrast difference lives entirely in the solvent |
| `sld_solvent` | almost never across contrasts — that is the variable; constrain each to its known value |
| `background` | rarely; incoherent background follows the sample composition |

A width shared across datasets shares the **width only**. Quadrature count,
truncation and distribution type stay per dataset, and relating widths with
different distribution types is refused — 0.15 of a lognormal and 0.15 of a
Schulz describe different distributions.

In [ ]:
mismatched = MultiFitter()
for seed, name in enumerate(('a', 'b'), start=501):
    mismatched.add(name, examples.simulate('sphere', radius=45, noise=0.02,
                                           seed=seed, npoints=30), model='sphere')
    mismatched[name].enable_polydispersity(True)

mismatched['a'].set_pd_param('radius', pd_width=0.1, pd_type='gaussian', vary=True)
mismatched['b'].set_pd_param('radius', pd_width=0.1, pd_type='lognormal', vary=True)

try:
    mismatched.share('radius_pd')
except ValueError as error:
    print(error)

## Where to go next

- [`docs/multifit.md`](../docs/multifit.md) — the reference for this API, including
  the full objective definition and the export format
- [`examples/simultaneous_fitting_example.py`](../examples/simultaneous_fitting_example.py) —
  the same four analyses as a script
- Not yet supported, and deliberately so until each has its own numerical tests:
  engines other than bumps, joint DREAM sampling, saving a joint analysis to
  JSON, inequality constraints, and batch fitting (which is a different
  operation — N independent fits, not one).